# Retail Sales Performance & Business Analytics

**Author:** Manpreet Jadon  
**Study period:** 2022–2025  
**Dataset:** Sales transactions and accompanying data dictionary

This notebook examines the supplied transaction records for sales, profit, customers, products, geography, and time trends. Every reported result is calculated from the source files at run time; no sample figures or assumed fields are embedded in the analysis.


## 1. Project Overview

This end-to-end analysis reviews the transaction data, documents quality considerations, creates a cleaned analysis copy, and explores business performance with reproducible tables and visualizations. The input CSV is treated as read-only.

## 2. Business Problem Statement

Retail decision-makers need a reliable view of how recorded sales and profit vary across products, customers, channels, locations, and time. The source may contain missing or inconsistent values, duplicates, and unusual measurements, so business comparisons should make their cleaning and metric definitions explicit.

## 3. Business Objectives

- Establish the dataset's coverage, structure, and data-quality limitations.
- Quantify recorded sales, profit, margin, customer/order activity, and flagged returns.
- Compare performance across available product, customer, channel, and geographic fields.
- Describe year-over-year and monthly/quarterly patterns using the transaction date.
- Surface potential outliers and numeric relationships for review, without silently deleting observations.
- Translate observed patterns into cautious, evidence-based actions.


## 4. Dataset Understanding

The supplied data dictionary is the authority for field meanings. The transaction file is loaded as provided, while a separate working copy is used for cleaning. Financial fields are analyzed as recorded; calculated formula checks are diagnostics, not replacements for those source values.

## 5. Data Dictionary Review

The next cell loads both files, compares the documented field list with the actual transaction headers, and displays the dictionary. Missing dictionary entries or extra transaction fields are reported rather than guessed.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
sns.set_theme(style="whitegrid", context="notebook", palette="deep")

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "Sales_transactions_2022_2025.csv"
DICTIONARY_PATH = PROJECT_DIR / "sales_data_dictionary.csv"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Transaction file not found: {DATA_PATH}")
if not DICTIONARY_PATH.is_file():
    raise FileNotFoundError(f"Data dictionary not found: {DICTIONARY_PATH}")

data_dictionary = pd.read_csv(DICTIONARY_PATH, encoding="utf-8-sig", dtype="string")
raw_df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    dtype={"Transaction_ID": "string", "Order_ID": "string", "Customer_ID": "string"},
)
original_source_shape = raw_df.shape

actual_columns = set(raw_df.columns)
documented_columns = set(data_dictionary["Column_Name"].dropna())
dictionary_review = pd.DataFrame(
    {
        "Column_Name": sorted(actual_columns | documented_columns),
    }
)
dictionary_review["In_transaction_file"] = dictionary_review["Column_Name"].isin(actual_columns)
dictionary_review["In_data_dictionary"] = dictionary_review["Column_Name"].isin(documented_columns)

print(f"Transaction file: {DATA_PATH.name}")
print(f"Dictionary file: {DICTIONARY_PATH.name}")
display(data_dictionary)
print("Dictionary-to-file coverage:")
display(dictionary_review)


## 6. Data Loading

The original dataframe is retained as `raw_df`. Cleaning and derived fields are created only in `df`; no source file is overwritten. Blank CSV fields are interpreted as missing values by pandas.

## 7. Dataset Shape and Structure

The following summary confirms the row/column count, header names, representative records, and dataframe memory/types before changes.


In [ ]:
print(f"Original shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]:,} columns")
print("Original column names:")
print(raw_df.columns.tolist())
display(raw_df.head())
raw_df.info()


## 8. Data Types and Data Quality Checks

The dictionary describes business-level types, while CSV loading may initially produce text for numeric and date values. This first-pass audit reports loaded types, unique-value counts, date parseability, ID uniqueness, and selected range/formula checks. A rule violation is a review flag, not proof that the source value is wrong.


In [ ]:
type_quality = pd.DataFrame(
    {
        "Loaded_dtype": raw_df.dtypes.astype(str),
        "Non_null": raw_df.notna().sum(),
        "Missing": raw_df.isna().sum(),
        "Distinct_non_null": raw_df.nunique(dropna=True),
    }
).sort_index()
display(type_quality)

date_check = pd.to_datetime(raw_df.get("Order_Date"), errors="coerce")
print(f"Order_Date parse failures: {date_check.isna().sum():,}")
if date_check.notna().any():
    print(f"Parsed date range: {date_check.min().date()} through {date_check.max().date()}")

if "Transaction_ID" in raw_df:
    print(
        "Transaction_ID duplicates (including repeated exact rows): "
        f"{raw_df['Transaction_ID'].duplicated(keep=False).sum():,} rows"
    )

numeric_candidates = [
    "Customer_Age", "Quantity", "Unit_Price", "Discount_Percentage",
    "Sales_Amount", "Cost_Amount", "Profit", "Delivery_Days",
    "Customer_Rating", "Inventory_Level", "Order_Year",
]
available_numeric_candidates = [column for column in numeric_candidates if column in raw_df]
numeric_preview = raw_df[available_numeric_candidates].apply(pd.to_numeric, errors="coerce")
display(numeric_preview.describe().T)

if {"Order_Date", "Order_Year"}.issubset(raw_df.columns):
    provided_year = pd.to_numeric(raw_df["Order_Year"], errors="coerce")
    parsed_year = date_check.dt.year
    year_mismatches = provided_year.notna() & parsed_year.notna() & provided_year.ne(parsed_year)
    print(f"Order_Year disagrees with parsed Order_Date: {year_mismatches.sum():,} rows")


## 9. Missing Value Analysis

Missingness is measured before any cleaning. It is not treated as zero, and rows are not dropped just because a field is missing. The reason for missingness is field-dependent; for example, return reasons may be blank when a return was not recorded.


In [ ]:
missing_summary = (
    raw_df.isna()
    .sum()
    .rename("Missing_Count")
    .to_frame()
    .assign(Missing_Percent=lambda table: table["Missing_Count"] / len(raw_df) * 100)
    .query("Missing_Count > 0")
    .sort_values("Missing_Count", ascending=False)
)
display(missing_summary)

if not missing_summary.empty:
    ax = missing_summary["Missing_Percent"].sort_values().plot(
        kind="barh", figsize=(9, max(4, 0.28 * len(missing_summary))), color="#4472C4"
    )
    ax.set(title="Missing values by field", xlabel="Rows missing (%)", ylabel="")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values were detected.")


## 10. Duplicate Analysis

The dictionary describes `Transaction_ID` as a unique transaction key. Exact duplicate rows are counted separately from repeated IDs with differing values. Only exact duplicate records will be removed from the analysis copy; non-identical ID collisions, if any, are surfaced for review and retained.


In [ ]:
exact_duplicate_mask = raw_df.duplicated(keep="first")
print(f"Exact duplicate rows beyond the first occurrence: {exact_duplicate_mask.sum():,}")
print(f"Rows participating in an exact duplicate: {raw_df.duplicated(keep=False).sum():,}")

id_duplicate_mask = raw_df["Transaction_ID"].duplicated(keep=False) if "Transaction_ID" in raw_df else pd.Series(False, index=raw_df.index)
id_duplicate_rows = raw_df.loc[id_duplicate_mask].sort_values("Transaction_ID")
if not id_duplicate_rows.empty:
    nonidentical_id_groups = id_duplicate_rows.drop_duplicates().groupby(
    "Transaction_ID", dropna=False
    ).size()
    nonidentical_ids = nonidentical_id_groups[nonidentical_id_groups > 1]
    print(f"Repeated Transaction_ID groups with differing rows: {len(nonidentical_ids):,}")
    if not nonidentical_ids.empty:
        display(id_duplicate_rows[id_duplicate_rows["Transaction_ID"].isin(nonidentical_ids.index)].head(20))


## 11. Data Cleaning

Cleaning is intentionally conservative: remove only fully identical duplicate records from the working copy, trim text whitespace, and normalize case-only variants in categorical labels by mapping each case-folded label to its most frequently observed spelling. This does not merge semantic aliases (for example, abbreviations and full names), impute missing values, correct suspected outliers, or recompute reported financial values. The original data remains available in `raw_df`.


In [ ]:
df = raw_df.drop_duplicates(keep="first").copy()
rows_removed_as_exact_duplicates = len(raw_df) - len(df)

text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
for column in text_columns:
    df[column] = df[column].str.strip().replace("", pd.NA)

categorical_candidates = [
    "Customer_Gender", "Customer_Segment", "Sales_Channel", "Store_Name",
    "Country", "Region", "City", "Product_Name", "Product_Category",
    "Product_Subcategory", "Payment_Method", "Order_Status",
    "Shipping_Method", "Return_Flag", "Return_Reason", "Sales_Representative",
    "Promotion_Code",
]
categorical_columns = [column for column in categorical_candidates if column in df]
for column in categorical_columns:
    non_missing = df[column].dropna()
    canonical_spelling = non_missing.groupby(non_missing.str.casefold()).agg(
        lambda values: values.value_counts().idxmax()
    )
    df[column] = df[column].map(
        lambda value: canonical_spelling.get(value.casefold(), value)
        if isinstance(value, str)
        else value
    )

numeric_columns = [
    "Customer_Age", "Quantity", "Unit_Price", "Discount_Percentage",
    "Sales_Amount", "Cost_Amount", "Profit", "Delivery_Days",
    "Customer_Rating", "Inventory_Level", "Order_Year",
]
for column in numeric_columns:
    if column in df:
        df[column] = pd.to_numeric(df[column], errors="coerce")

if "Order_Date" in df:
    df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
if "Order_Time" in df:
    parsed_time = pd.to_datetime(df["Order_Time"], format="%H:%M:%S", errors="coerce")
    df["Order_Hour"] = parsed_time.dt.hour.astype("Int64")

print(f"Analysis-copy shape after exact duplicate removal: {df.shape}")
print(f"Exact duplicate rows removed from analysis copy: {rows_removed_as_exact_duplicates:,}")
print(f"Source dataframe shape unchanged: {raw_df.shape == original_source_shape}")
print("Remaining repeated transaction IDs:", df["Transaction_ID"].duplicated().sum() if "Transaction_ID" in df else "field unavailable")


## 12. Date/Time Feature Extraction

Calendar features are derived from parsed `Order_Date`, rather than relying on the convenience `Order_Year` field. Date-derived year is used for time-series analysis; the supplied year is retained for comparison. Invalid dates remain missing and are excluded only from date-based summaries.


In [ ]:
if "Order_Date" in df:
    df["Order_Year_Calc"] = df["Order_Date"].dt.year.astype("Int64")
    df["Order_Month"] = df["Order_Date"].dt.month.astype("Int64")
    df["Order_Month_Name"] = df["Order_Date"].dt.month_name()
    df["Order_Quarter"] = df["Order_Date"].dt.to_period("Q").astype("string")
    df["Order_Year_Month"] = df["Order_Date"].dt.to_period("M").astype("string")

date_features = [
    column for column in
    ["Order_Date", "Order_Time", "Order_Hour", "Order_Year_Calc", "Order_Month", "Order_Month_Name", "Order_Quarter", "Order_Year_Month"]
    if column in df
]
display(df[date_features].head())


## 13. Exploratory Data Analysis

All charts and grouped tables below use the cleaned analysis copy. The case-only category standardization improves label consistency but intentionally does not reconcile different geographic aliases. Interpret comparisons with the remaining missing values and data-quality flags in mind.


In [ ]:
print(f"Analysis rows: {len(df):,}; fields: {df.shape[1]:,}")
if "Order_Date" in df and df["Order_Date"].notna().any():
    print(
        "Analysis period:",
        df["Order_Date"].min().date(),
        "to",
        df["Order_Date"].max().date(),
    )

for column in ["Order_Status", "Sales_Channel", "Product_Category", "Customer_Segment", "Return_Flag"]:
    if column in df:
        print(f"\n{column} — transaction rows")
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame().head(15))


## 14. Sales and Revenue Analysis

`Sales_Amount` and `Profit` are summed as recorded in the file. These transaction-level totals include every retained row, including rows marked returned, cancelled, processing, or shipped; status-specific views are shown separately so users can choose a policy appropriate to their accounting definition. Margin is total recorded profit divided by total recorded sales, not an average of row margins.


In [ ]:
required_financial_fields = {"Sales_Amount", "Profit", "Cost_Amount", "Order_ID"}
if required_financial_fields.issubset(df.columns):
    total_sales = df["Sales_Amount"].sum(min_count=1)
    total_profit = df["Profit"].sum(min_count=1)
    total_cost = df["Cost_Amount"].sum(min_count=1)
    unique_orders = df["Order_ID"].nunique(dropna=True)
    average_order_value = (
        df.groupby("Order_ID", dropna=True)["Sales_Amount"].sum().mean()
        if unique_orders
        else np.nan
    )
    kpi_table = pd.DataFrame(
        {
            "Metric": [
                "Recorded sales amount",
                "Recorded profit",
                "Recorded cost",
                "Profit / sales margin",
                "Transaction rows in analysis copy",
                "Distinct orders",
                "Distinct customers",
                "Average recorded sales per order",
            ],
            "Value": [
                total_sales,
                total_profit,
                total_cost,
                total_profit / total_sales if total_sales else np.nan,
                len(df),
                unique_orders,
                df["Customer_ID"].nunique(dropna=True) if "Customer_ID" in df else np.nan,
                average_order_value,
            ],
        }
    )
    display(kpi_table)

if "Order_Status" in df and "Sales_Amount" in df:
    status_sales = (
        df.groupby("Order_Status", dropna=False)
        .agg(Transaction_Rows=("Sales_Amount", "size"), Recorded_Sales=("Sales_Amount", "sum"), Recorded_Profit=("Profit", "sum"))
        .sort_values("Recorded_Sales", ascending=False)
    )
    display(status_sales)


## 15. Product Performance Analysis

Category and product summaries show recorded sales, profit, volume, and margin where available. Missing product labels are not inferred from product IDs; they remain excluded from named product groups.


In [ ]:
if "Product_Category" in df and "Sales_Amount" in df:
    category_metrics = (
        df.groupby("Product_Category", dropna=False)
        .agg(
            Transaction_Rows=("Sales_Amount", "size"),
            Recorded_Sales=("Sales_Amount", "sum"),
            Recorded_Profit=("Profit", "sum"),
            Units=("Quantity", "sum") if "Quantity" in df else ("Sales_Amount", "size"),
        )
        .sort_values("Recorded_Sales", ascending=False)
    )
    category_metrics["Profit_Margin"] = category_metrics["Recorded_Profit"] / category_metrics["Recorded_Sales"]
    display(category_metrics)

if {"Product_ID", "Product_Name", "Sales_Amount", "Profit"}.issubset(df.columns):
    product_metrics = (
        df.groupby(["Product_ID", "Product_Name"], dropna=False)
        .agg(
            Transaction_Rows=("Sales_Amount", "size"),
            Recorded_Sales=("Sales_Amount", "sum"),
            Recorded_Profit=("Profit", "sum"),
            Units=("Quantity", "sum") if "Quantity" in df else ("Sales_Amount", "size"),
        )
        .sort_values("Recorded_Sales", ascending=False)
    )
    display(product_metrics.head(15))


## 16. Customer Analysis

Customer summaries are keyed on `Customer_ID`, not display name. Customer name and demographic fields may be missing or shared, so they are not treated as unique identifiers. The tables distinguish transaction-line activity from customer-level aggregation.


In [ ]:
if {"Customer_ID", "Sales_Amount"}.issubset(df.columns):
    customer_metrics = (
        df.groupby("Customer_ID", dropna=True)
        .agg(
            Transaction_Rows=("Sales_Amount", "size"),
            Recorded_Sales=("Sales_Amount", "sum"),
            Recorded_Profit=("Profit", "sum"),
            Distinct_Orders=("Order_ID", "nunique") if "Order_ID" in df else ("Sales_Amount", "size"),
        )
        .sort_values("Recorded_Sales", ascending=False)
    )
    print(f"Customer IDs represented: {len(customer_metrics):,}")
    display(customer_metrics.head(15))

for column in ["Customer_Segment", "Customer_Gender"]:
    if column in df and "Sales_Amount" in df:
        segmentation = (
            df.groupby(column, dropna=False)
            .agg(Transaction_Rows=("Sales_Amount", "size"), Recorded_Sales=("Sales_Amount", "sum"), Recorded_Profit=("Profit", "sum"))
            .sort_values("Recorded_Sales", ascending=False)
        )
        display(segmentation)


## 17. Regional/Geographical Analysis

Country, region, and city are available in the supplied file. Results are shown at country and region level; differently named aliases are not silently combined. Check the dictionary and source labels before making operational decisions from regional rankings.


In [ ]:
for geo_column in ["Country", "Region", "City"]:
    if geo_column in df and "Sales_Amount" in df:
        geography = (
            df.groupby(geo_column, dropna=False)
            .agg(Transaction_Rows=("Sales_Amount", "size"), Recorded_Sales=("Sales_Amount", "sum"), Recorded_Profit=("Profit", "sum"))
            .sort_values("Recorded_Sales", ascending=False)
        )
        print(f"Performance by {geo_column}")
        display(geography.head(20))


## 18. Year-over-Year Sales Trends

The reported date-derived calendar year is used for the comparison. Year-over-year change is calculated only when a prior-year total exists and is not interpreted as a like-for-like growth measure beyond the coverage in the file.


In [ ]:
yearly_metrics = pd.DataFrame()
if {"Order_Year_Calc", "Sales_Amount"}.issubset(df.columns):
    yearly_metrics = (
        df.dropna(subset=["Order_Year_Calc"])
        .groupby("Order_Year_Calc")
        .agg(
            Transaction_Rows=("Sales_Amount", "size"),
            Recorded_Sales=("Sales_Amount", "sum"),
            Recorded_Profit=("Profit", "sum"),
            Distinct_Orders=("Order_ID", "nunique") if "Order_ID" in df else ("Sales_Amount", "size"),
        )
        .sort_index()
    )
    yearly_metrics["Sales_YoY_Percent"] = yearly_metrics["Recorded_Sales"].pct_change() * 100
    yearly_metrics["Profit_Margin"] = yearly_metrics["Recorded_Profit"] / yearly_metrics["Recorded_Sales"]
    display(yearly_metrics)

if not yearly_metrics.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    yearly_metrics["Recorded_Sales"].plot(kind="bar", ax=ax, color="#4472C4")
    ax.set(title="Recorded sales by calendar year", xlabel="Order year", ylabel="Recorded sales amount")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()


## 19. Monthly/Quarterly Sales Trends

Monthly totals support seasonality exploration across the observed period. Quarterly totals are also provided. Partial or missing date coverage would affect comparability, so confirm the date range before treating a seasonal pattern as recurring.


In [ ]:
monthly_sales = pd.Series(dtype="float64")
if {"Order_Date", "Sales_Amount"}.issubset(df.columns):
    monthly_sales = (
        df.dropna(subset=["Order_Date"])
        .set_index("Order_Date")["Sales_Amount"]
        .resample("MS")
        .sum()
    )
    fig, ax = plt.subplots(figsize=(13, 5))
    monthly_sales.plot(ax=ax, color="#2F75B5", linewidth=1.8)
    ax.set(title="Monthly recorded sales", xlabel="Order month", ylabel="Recorded sales amount")
    plt.tight_layout()
    plt.show()

    quarterly_sales = (
        df.dropna(subset=["Order_Date"])
        .set_index("Order_Date")["Sales_Amount"]
        .resample("QS")
        .sum()
        .rename("Recorded_Sales")
    )
    display(quarterly_sales.to_frame().tail(20))


## 20. KPI Analysis

This section expands on the headline totals with channel and return-flag measures. Return rate is the share of transaction rows marked `Yes`, not the share of orders or sales value. These operational indicators should not be interpreted as accounting-adjusted net revenue without a documented status policy.


In [ ]:
if {"Sales_Channel", "Sales_Amount"}.issubset(df.columns):
    channel_kpis = (
        df.groupby("Sales_Channel", dropna=False)
        .agg(Transaction_Rows=("Sales_Amount", "size"), Recorded_Sales=("Sales_Amount", "sum"), Recorded_Profit=("Profit", "sum"))
        .sort_values("Recorded_Sales", ascending=False)
    )
    channel_kpis["Sales_Share_Percent"] = channel_kpis["Recorded_Sales"] / channel_kpis["Recorded_Sales"].sum() * 100
    channel_kpis["Profit_Margin"] = channel_kpis["Recorded_Profit"] / channel_kpis["Recorded_Sales"]
    display(channel_kpis)

if "Return_Flag" in df:
    return_rows = df["Return_Flag"].astype("string").str.casefold().eq("yes").sum()
    return_rate = return_rows / len(df) if len(df) else np.nan
    print(f"Rows flagged as returned: {return_rows:,} ({return_rate:.2%} of analysis rows)")
    if "Sales_Amount" in df:
        display(
            df.groupby("Return_Flag", dropna=False)
            .agg(Transaction_Rows=("Sales_Amount", "size"), Recorded_Sales=("Sales_Amount", "sum"), Recorded_Profit=("Profit", "sum"))
        )


## 21. Outlier Analysis

The IQR rule is used to flag unusually low/high observations in selected numeric fields. It is a screening heuristic, not a deletion rule: returned quantities, genuine large orders, and data-entry errors can all appear extreme. Range checks are shown alongside distribution plots.


In [ ]:
outlier_columns = [column for column in ["Sales_Amount", "Profit", "Quantity", "Customer_Age"] if column in df]
outlier_rows = []
for column in outlier_columns:
    values = df[column].dropna()
    if values.empty:
        continue
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = values.lt(lower) | values.gt(upper)
    outlier_rows.append(
        {
            "Field": column,
            "Q1": q1,
            "Q3": q3,
            "IQR_Lower_Bound": lower,
            "IQR_Upper_Bound": upper,
            "Flagged_Count": int(flagged.sum()),
            "Flagged_Percent": flagged.mean() * 100,
        }
    )
outlier_summary = pd.DataFrame(outlier_rows).set_index("Field") if outlier_rows else pd.DataFrame()
display(outlier_summary)

if outlier_columns:
    fig, axes = plt.subplots(len(outlier_columns), 1, figsize=(10, 2.8 * len(outlier_columns)))
    axes = np.atleast_1d(axes)
    for ax, column in zip(axes, outlier_columns):
        sns.boxplot(x=df[column], ax=ax, color="#9DC3E6")
        ax.set_title(f"{column} distribution (IQR flags retained)")
    plt.tight_layout()
    plt.show()

range_flags = {}
if "Customer_Age" in df:
    range_flags["Customer_Age outside 0–100"] = int((df["Customer_Age"].lt(0) | df["Customer_Age"].gt(100)).sum())
if "Quantity" in df:
    range_flags["Negative Quantity"] = int(df["Quantity"].lt(0).sum())
if "Discount_Percentage" in df:
    range_flags["Discount_Percentage outside 0–100"] = int(
        (df["Discount_Percentage"].lt(0) | df["Discount_Percentage"].gt(100)).sum()
    )
if "Customer_Rating" in df:
    range_flags["Customer_Rating outside 1–5"] = int(
        (df["Customer_Rating"].lt(1) | df["Customer_Rating"].gt(5)).sum()
    )
display(pd.Series(range_flags, name="Rows flagged").to_frame())


## 22. Correlation Analysis

Pearson correlations are calculated for available numeric measures. Correlation describes linear association only and is sensitive to outliers; it does not establish causation. Derived calendar fields and identifiers are excluded.


In [ ]:
correlation_candidates = [
    "Quantity", "Unit_Price", "Discount_Percentage", "Sales_Amount",
    "Cost_Amount", "Profit", "Customer_Age", "Delivery_Days",
    "Customer_Rating", "Inventory_Level",
]
correlation_columns = [column for column in correlation_candidates if column in df]
correlation_matrix = df[correlation_columns].corr(min_periods=2)
display(correlation_matrix)
if not correlation_matrix.empty:
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(correlation_matrix, cmap="vlag", center=0, annot=True, fmt=".2f", square=True, ax=ax)
    ax.set_title("Correlation among selected numeric fields")
    plt.tight_layout()
    plt.show()


## 23. Business-Focused Visualizations

These views compare category, channel, geography, and customer-segment sales on a consistent scale. The ranking charts are descriptive, and all amounts retain the currency/unit convention of the source file (which does not document a currency code).


In [ ]:
plot_specs = [
    ("Product_Category", "Recorded sales by product category"),
    ("Sales_Channel", "Recorded sales by sales channel"),
    ("Country", "Recorded sales by country"),
    ("Customer_Segment", "Recorded sales by customer segment"),
]
available_specs = [(column, title) for column, title in plot_specs if column in df and "Sales_Amount" in df]
fig, axes = plt.subplots(1, len(available_specs), figsize=(6 * max(1, len(available_specs)), 5))
axes = np.atleast_1d(axes)
for ax, (column, title) in zip(axes, available_specs):
    plot_data = (
        df.groupby(column, dropna=False)["Sales_Amount"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .sort_values()
    )
    plot_data.plot(kind="barh", ax=ax, color="#5B9BD5")
    ax.set(title=title, xlabel="Recorded sales amount", ylabel="")
plt.tight_layout()
plt.show()

if {"Product_Category", "Order_Year_Calc", "Sales_Amount"}.issubset(df.columns):
    category_year = df.pivot_table(
        index="Order_Year_Calc", columns="Product_Category", values="Sales_Amount", aggfunc="sum"
    )
    ax = category_year.plot(kind="line", marker="o", figsize=(10, 5))
    ax.set(title="Category sales by calendar year", xlabel="Order year", ylabel="Recorded sales amount")
    plt.tight_layout()
    plt.show()


## 24. Key Findings and Insights

The statements below are generated from this file at execution time. They describe recorded transaction totals and rankings, not causal explanations. The actual currency is not specified in the supplied dictionary, so values are reported without a currency symbol.


In [ ]:
insights = []
if "Sales_Amount" in df and not df.empty:
    total_sales_value = df["Sales_Amount"].sum()
    insights.append(f"- The analysis copy contains **{len(df):,} transaction rows** with **{df['Order_ID'].nunique():,} distinct orders** and **{df['Customer_ID'].nunique():,} distinct customer IDs**.")
    insights.append(f"- Recorded sales total **{total_sales_value:,.2f}**; recorded profit totals **{df['Profit'].sum():,.2f}**.")

if not yearly_metrics.empty:
    valid_growth = yearly_metrics["Sales_YoY_Percent"].dropna()
    if not valid_growth.empty:
        latest_year = valid_growth.index[-1]
        latest_growth = valid_growth.iloc[-1]
        direction = "increased" if latest_growth >= 0 else "decreased"
        insights.append(f"- From {int(latest_year) - 1} to {int(latest_year)}, recorded sales {direction} by **{abs(latest_growth):.2f}%** ({latest_growth:+.2f}%).")
    top_year = yearly_metrics["Recorded_Sales"].idxmax()
    insights.append(f"- The highest recorded annual sales total occurred in **{int(top_year)}**.")

for grouping_column, label in [("Product_Category", "category"), ("Sales_Channel", "channel"), ("Country", "country")]:
    if grouping_column in df and "Sales_Amount" in df:
        grouped_sales = df.groupby(grouping_column)["Sales_Amount"].sum().dropna()
        if not grouped_sales.empty:
            leader = grouped_sales.idxmax()
            share = grouped_sales.max() / grouped_sales.sum() * 100
            insights.append(f"- **{leader}** leads recorded sales by {label}, representing **{share:.1f}%** of the named-group total.")

if "Return_Flag" in df:
    return_share = df["Return_Flag"].astype("string").str.casefold().eq("yes").mean() * 100
    insights.append(f"- **{return_share:.2f}%** of analysis transaction rows are marked `Return_Flag = Yes`; this is a row-based rate, not an order- or value-based rate.")

display(Markdown("\n".join(insights) if insights else "No available financial fields were found to generate findings."))


## 25. Business Recommendations

Recommendations are conditional on the observed patterns and should be validated with finance, operations, and data owners before changing policy. The source does not define currency, net-revenue treatment by order status, or whether negative quantities are expected for returns.


In [ ]:
recommendations = []
if rows_removed_as_exact_duplicates:
    recommendations.append(
        f"Investigate the upstream process responsible for {rows_removed_as_exact_duplicates:,} fully duplicated rows and enforce transaction-key uniqueness at ingestion."
    )
if "Discount_Percentage" in df:
    invalid_discount_count = int((df["Discount_Percentage"].lt(0) | df["Discount_Percentage"].gt(100)).sum())
    if invalid_discount_count:
        recommendations.append(
            f"Review {invalid_discount_count:,} discount value(s) outside 0–100 before using discount measures for promotion policy; do not silently cap them."
        )
if "Customer_Age" in df:
    invalid_age_count = int((df["Customer_Age"].lt(0) | df["Customer_Age"].gt(100)).sum())
    if invalid_age_count:
        recommendations.append(
            f"Validate {invalid_age_count:,} age value(s) outside 0–100 with the source system; keep them flagged rather than imputing or deleting."
        )
if "Quantity" in df:
    negative_quantity_count = int(df["Quantity"].lt(0).sum())
    if negative_quantity_count:
        recommendations.append(
            f"Confirm whether {negative_quantity_count:,} negative-quantity row(s) represent returns/adjustments or errors, then document the agreed treatment for volume reporting."
        )
if not yearly_metrics.empty and len(yearly_metrics) > 1:
    last_growth = yearly_metrics["Sales_YoY_Percent"].dropna()
    if not last_growth.empty:
        latest_year = last_growth.index[-1]
        latest_change = last_growth.iloc[-1]
        if latest_change < 0:
            recommendations.append(
                f"Investigate the drivers of the {abs(latest_change):.2f}% recorded-sales decline in {int(latest_year)} versus {int(latest_year) - 1} by decomposing volume, price, product mix, channel, and status."
            )
        else:
            recommendations.append(
                f"Validate whether the {latest_change:.2f}% recorded-sales increase in {int(latest_year)} is supported by comparable order status, volume, and product/channel mix."
            )
if "Order_Status" in df:
    recommendations.append(
        "Agree a finance-approved inclusion rule for Completed, Returned, Cancelled, and in-progress orders before publishing an accounting revenue KPI."
    )
if {"Region", "Sales_Amount"}.issubset(df.columns):
    recommendations.append(
        "Standardize geographic aliases in the source system before using region-level rankings for resource allocation."
    )
recommendations.append(
    "Add a documented currency code and clarify whether Sales_Amount is net of returns/discounts; the supplied dictionary does not specify these conventions."
)

display(Markdown("\n".join(f"- {item}" for item in recommendations)))


## 26. Final Conclusion

This notebook provides a reproducible view of the supplied 2022–2025 sales records, preserves the original data, reports its limitations, and calculates descriptive sales, product, customer, geographic, temporal, and KPI summaries. Exact duplicates are excluded only from the analysis copy; missing values and unusual measurements remain visible. Business decisions should use an agreed revenue/status policy and resolve data-quality questions with the source owners before treating these descriptive results as audited financial figures.

---

**Author:** Manpreet Jadon
